In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("../dataset/monthly_spending.csv", parse_dates=["month"])
df = df.sort_values(["user_id", "category", "month"]).reset_index(drop=True)

g = df.groupby(["user_id", "category"])["amount"]
df["last_month"]    = g.shift(1)                                           # same as last month
df["avg_3m"]        = g.transform(lambda s: s.shift(1).rolling(3).mean())  # average of last 3 months
df["avg_12m"]       = g.transform(lambda s: s.shift(1).rolling(12).mean()) # average of last 12 months
df["same_month_ly"] = g.shift(12)                                          # same month, one year ago

BASELINES = ["last_month", "avg_3m", "avg_12m", "same_month_ly"]
test = df[df["month"] >= "2026-05-01"].copy()
print(len(test), "test rows | missing baseline values:", test[BASELINES].isnull().sum().sum())

1980 test rows | missing baseline values: 0


In [2]:
def wape(y, p):
    return np.abs(y - p).sum() / y.sum()

def mae(y, p):
    return np.abs(y - p).mean()

In [3]:
rows = [{"baseline": b,
         "MAE (INR)": round(mae(test["amount"], test[b])),
         "WAPE": round(wape(test["amount"], test[b]), 4)} for b in BASELINES]
pd.DataFrame(rows)

,baseline,MAE (INR),WAPE
0,last_month,1404,0.2156
1,avg_3m,1239,0.1902
2,avg_12m,1192,0.1830
3,same_month_ly,1700,0.2610


In [4]:
by_cat = test.groupby("category").apply(
    lambda d: pd.Series({b: round(wape(d["amount"], d[b]), 3) for b in BASELINES})
)
by_cat

,last_month,avg_3m,avg_12m,same_month_ly
category,,,,
Bills & Utilities,0.154,0.183,0.149,0.135
Entertainment,0.273,0.212,0.185,0.248
Food,0.137,0.113,0.107,0.143
Groceries,0.097,0.087,0.084,0.114
Health,0.482,0.396,0.343,0.473
Other,0.464,0.415,0.360,0.512
Personal Care,0.294,0.237,0.221,0.268
Rent,0.008,0.014,0.036,0.062
Shopping,0.463,0.389,0.371,0.512


In [5]:
tot = test.groupby(["user_id", "month"])[["amount"] + BASELINES].sum()
{b: round(wape(tot["amount"], tot[b]), 4) for b in BASELINES}

{'last_month': np.float64(0.1013),
 'avg_3m': np.float64(0.0904),
 'avg_12m': np.float64(0.0856),
 'same_month_ly': np.float64(0.1297)}

In [6]:
test.groupby("month").apply(lambda d: round(wape(d["amount"], d["avg_3m"]), 3))

month
2026-05-01    0.175
2026-06-01    0.183
2026-07-01    0.195
2026-08-01    0.164
2026-09-01    0.168
2026-10-01    0.246
dtype: float64